In [83]:
import pandas as pd

# Load the dataset
file_path = '/Users/lkimball/Desktop/Wesleyan_Tennis_Analytics/singles_matches_df.csv'
data = pd.read_csv(file_path)

# Display the first few rows of the dataframe and the column names
data.head()

,Name,Position,Duration,1 - Event,2 - Player Name,3 - Opponent Name,4 - Surface,A1: Server,A2: Serve Data,A3: Serve Placement,...,x: Player X,x: Player Y,zz - Rally Data,zz - Returner,J4: Poach Returning Team,F5: Point Penalty,H3: Court Position,x: Player A Set Score,x: Player B Set Score,z - 1st Serve Fault Target
0,0-0 (1),0,13033,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0-15 (1),13033,15433,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,2nd Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15-15 (1),28467,11200,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Body,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15-30 (1),39667,17800,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30-30 (1),57467,7700,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Body,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
# List of players provided by the user
players = [
    "Ben Mitchell", "Leila Epstein", "Lane Durkin", "Jan Hrazdil", "Sarah Youngberg", 
    "Chase Kasday", "Alexander Bernard", "Patrick Kypson", "Carlos Alcaraz Garfia", 
    "Jackie Soloveychik", "JT Bilski", "Max Lustgarten", "Harrison Collomb", 
    "Nicolas Kotzen", "Brett Keeling", "Michael Zheng", "Alexandra Burnett", 
    "Caitlyn Ferrante", 'Eudice Chong'
]

# Filter data for these players
filtered_data = data[data['2 - Player Name'].isin(players)]

# Total points for each player
total_points = filtered_data['2 - Player Name'].value_counts().rename('Total Points')

# Serve +1 points for each player
Op_ret_serv_points = filtered_data[(filtered_data['A1: Server'].isin(players)) & 
                                      (filtered_data['B2: Return Data'].str.contains('Made|Winner'))]['A1: Server'].value_counts().rename('Opponent Returned Serve')

# Combine the data
points_summary = pd.DataFrame([total_points, Op_ret_serv_points]).T.fillna(0).astype(int)
points_summary.reset_index().rename(columns={'index': 'Player Name'})


,Player Name,Total Points,Opponent Returned Serve
0,Eudice Chong,1104,430
1,Leila Epstein,749,283
2,Ben Mitchell,601,191
3,JT Bilski,572,216
4,Brett Keeling,489,162
5,Harrison Collomb,435,162
6,Lane Durkin,363,116
7,Sarah Youngberg,359,123
8,Alexander Bernard,326,128
9,Michael Zheng,322,121


In [85]:
# Serve +1 in play points
serve_plus_one_in_play = filtered_data[(filtered_data['A1: Server'].isin(players)) & 
                                       (filtered_data['C2: Serve +1 Data'].str.contains('In Play', na=False))]['A1: Server'].value_counts().rename('Serve +1 In Play')

# Serve +1 winners points
serve_plus_one_winners = filtered_data[(filtered_data['A1: Server'].isin(players)) & 
                                       (filtered_data['C2: Serve +1 Data'].str.contains('Winner', na=False))]['A1: Server'].value_counts().rename('Serve +1 Winners')

# Serve +1 errors points
serve_plus_one_errors = filtered_data[(filtered_data['A1: Server'].isin(players)) & 
                                      (filtered_data['C2: Serve +1 Data'].str.contains('Error', na=False))]['A1: Server'].value_counts().rename('Serve +1 Errors')

# Add new columns to the summary
points_summary = points_summary.join([serve_plus_one_in_play, serve_plus_one_winners, serve_plus_one_errors]).fillna(0).astype(int)
points_summary.reset_index().rename(columns={'index': 'Player Name'})


,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors
0,Eudice Chong,1104,430,334,27,63
1,Leila Epstein,749,283,210,15,53
2,Ben Mitchell,601,191,162,8,18
3,JT Bilski,572,216,175,13,26
4,Brett Keeling,489,162,126,12,19
5,Harrison Collomb,435,162,114,4,43
6,Lane Durkin,363,116,95,6,12
7,Sarah Youngberg,359,123,98,4,21
8,Alexander Bernard,326,128,99,16,10
9,Michael Zheng,322,121,96,7,15


In [86]:
# Calculate total Serve +1 actions as the denominator
points_summary['Total Serve +1 Actions'] = points_summary['Serve +1 In Play'] + points_summary['Serve +1 Winners'] + points_summary['Serve +1 Errors']

# Calculate Serve +1 Winner % and Serve +1 Error %
points_summary['Serve +1 Winner %'] = (points_summary['Serve +1 Winners'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['Serve +1 Error %'] = (points_summary['Serve +1 Errors'] / points_summary['Total Serve +1 Actions']) * 100

# Display the updated summary with percentages
updated_summary = points_summary.reset_index().rename(columns={'index': 'Player Name'})
updated_summary[['Player Name', 'Total Points', 'Opponent Returned Serve', 'Serve +1 In Play', 
                 'Serve +1 Winners', 'Serve +1 Errors', 'Serve +1 Winner %', 'Serve +1 Error %']]


,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Serve +1 Winner %,Serve +1 Error %
0,Eudice Chong,1104,430,334,27,63,6.367925,14.858491
1,Leila Epstein,749,283,210,15,53,5.395683,19.064748
2,Ben Mitchell,601,191,162,8,18,4.255319,9.574468
3,JT Bilski,572,216,175,13,26,6.074766,12.149533
4,Brett Keeling,489,162,126,12,19,7.643312,12.101911
5,Harrison Collomb,435,162,114,4,43,2.484472,26.708075
6,Lane Durkin,363,116,95,6,12,5.309735,10.619469
7,Sarah Youngberg,359,123,98,4,21,3.252033,17.073171
8,Alexander Bernard,326,128,99,16,10,12.800000,8.000000
9,Michael Zheng,322,121,96,7,15,5.932203,12.711864


In [87]:
# Opponent Return +1 Errors
opponent_return_plus_one_errors = filtered_data[(filtered_data['A1: Server'].isin(players)) & 
                                                (filtered_data['D2: Return +1 Data'].str.contains('Error', na=False))]['A1: Server'].value_counts().rename('Opponent Return +1 Errors')

# Add this new column to the summary
points_summary = points_summary.join(opponent_return_plus_one_errors).fillna(0).astype(int)

# Display the updated summary
updated_summary = points_summary.reset_index().rename(columns={'index': 'Player Name'})
updated_summary[['Player Name', 'Total Points', 'Opponent Returned Serve', 'Serve +1 In Play', 
                 'Serve +1 Winners', 'Serve +1 Errors', 'Opponent Return +1 Errors']]


,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Opponent Return +1 Errors
0,Eudice Chong,1104,430,334,27,63,53
1,Leila Epstein,749,283,210,15,53,31
2,Ben Mitchell,601,191,162,8,18,34
3,JT Bilski,572,216,175,13,26,26
4,Brett Keeling,489,162,126,12,19,19
5,Harrison Collomb,435,162,114,4,43,26
6,Lane Durkin,363,116,95,6,12,26
7,Sarah Youngberg,359,123,98,4,21,24
8,Alexander Bernard,326,128,99,16,10,26
9,Michael Zheng,322,121,96,7,15,15


In [88]:
# Adjust filter for Serve +1 FH Winners and Errors based on the example provided
serve_plus_one_fh_winners = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Forehand') &
    (filtered_data['C2: Serve +1 Data'] == 'Winner')
]['A1: Server'].value_counts().rename('S+1 FH Winners')

serve_plus_one_fh_errors = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Forehand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error')
]['A1: Server'].value_counts().rename('S+1 FH Errors')

# Update summary table with the correct counts
points_summary['S+1 FH Winners'] = serve_plus_one_fh_winners
points_summary['S+1 FH Errors'] = serve_plus_one_fh_errors

# Fill NaN values with 0 and convert to int
points_summary = points_summary.fillna(0).astype(int)

# Recalculate percentages relative to total serve +1 actions
points_summary['S+1 FH Winners %'] = (points_summary['S+1 FH Winners'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors %'] = (points_summary['S+1 FH Errors'] / points_summary['Total Serve +1 Actions']) * 100

# Display the updated summary with FH Winners and Errors
updated_summary = points_summary.reset_index().rename(columns={'index': 'Player Name'})
updated_summary[['Player Name', 'Total Points', 'Opponent Returned Serve', 'Serve +1 In Play', 
                 'S+1 FH Winners', 'S+1 FH Winners %', 'S+1 FH Errors', 'S+1 FH Errors %']]


,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,S+1 FH Winners,S+1 FH Winners %,S+1 FH Errors,S+1 FH Errors %
0,Eudice Chong,1104,430,334,11,2.594340,39,9.198113
1,Leila Epstein,749,283,210,6,2.158273,37,13.309353
2,Ben Mitchell,601,191,162,0,0.000000,11,5.851064
3,JT Bilski,572,216,175,7,3.271028,13,6.074766
4,Brett Keeling,489,162,126,5,3.184713,9,5.732484
5,Harrison Collomb,435,162,114,2,1.242236,25,15.527950
6,Lane Durkin,363,116,95,4,3.539823,5,4.424779
7,Sarah Youngberg,359,123,98,2,1.626016,18,14.634146
8,Alexander Bernard,326,128,99,5,4.000000,5,4.000000
9,Michael Zheng,322,121,96,0,0.000000,8,6.779661


In [89]:
# Filter data for S+1 FH Errors into the net, wide, and long based on Serve +1 columns
fh_errors_net = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Forehand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Net', na=False))
]['A1: Server'].value_counts().rename('S+1 FH Errors Net')

fh_errors_wide = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Forehand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Wide', na=False))
]['A1: Server'].value_counts().rename('S+1 FH Errors Wide')

fh_errors_long = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Forehand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Long', na=False))
]['A1: Server'].value_counts().rename('S+1 FH Errors Long')

# Update summary table with the correct counts
points_summary['S+1 FH Errors Net'] = fh_errors_net
points_summary['S+1 FH Errors Wide'] = fh_errors_wide
points_summary['S+1 FH Errors Long'] = fh_errors_long

# Fill NaN values with 0 and convert to int
points_summary = points_summary.fillna(0).astype(int)

# Calculate total FH Errors
points_summary['S+1 FH Errors'] = points_summary['S+1 FH Errors Net'] + points_summary['S+1 FH Errors Wide'] + points_summary['S+1 FH Errors Long']

# Recalculate percentages relative to total serve +1 actions
points_summary['S+1 FH Errors Net %'] = (points_summary['S+1 FH Errors Net'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors Wide %'] = (points_summary['S+1 FH Errors Wide'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors Long %'] = (points_summary['S+1 FH Errors Long'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors %'] = (points_summary['S+1 FH Errors'] / points_summary['Total Serve +1 Actions']) * 100

# Display the updated summary with FH Errors
updated_summary = points_summary.reset_index().rename(columns={'index': 'Player Name'})
updated_summary[['Player Name', 'S+1 FH Errors', 'S+1 FH Errors Net', 'S+1 FH Errors Net %', 
                 'S+1 FH Errors Wide', 'S+1 FH Errors Wide %', 'S+1 FH Errors Long', 'S+1 FH Errors Long %']]


,Player Name,S+1 FH Errors,S+1 FH Errors Net,S+1 FH Errors Net %,S+1 FH Errors Wide,S+1 FH Errors Wide %,S+1 FH Errors Long,S+1 FH Errors Long %
0,Eudice Chong,39,11,2.594340,10,2.358491,18,4.245283
1,Leila Epstein,37,8,2.877698,8,2.877698,21,7.553957
2,Ben Mitchell,11,2,1.063830,5,2.659574,4,2.127660
3,JT Bilski,13,5,2.336449,3,1.401869,5,2.336449
4,Brett Keeling,9,1,0.636943,1,0.636943,7,4.458599
5,Harrison Collomb,25,11,6.832298,7,4.347826,7,4.347826
6,Lane Durkin,5,4,3.539823,0,0.000000,1,0.884956
7,Sarah Youngberg,18,4,3.252033,6,4.878049,8,6.504065
8,Alexander Bernard,5,1,0.800000,1,0.800000,3,2.400000
9,Michael Zheng,8,2,1.694915,3,2.542373,3,2.542373


In [90]:
# Ensure data frames have a common index to avoid overlap when joining
# Set the index of the points_summary to match the indexes used in the error data counts
points_summary.set_index(updated_summary['Player Name'], inplace=True)

# Update summary table with these new counts correctly
points_summary['S+1 FH Errors Net'] = fh_errors_net
points_summary['S+1 FH Errors Wide'] = fh_errors_wide
points_summary['S+1 FH Errors Long'] = fh_errors_long
points_summary.fillna(0, inplace=True)
points_summary = points_summary.astype(int)

# Recalculate percentages relative to total S+1 FH Errors
points_summary['S+1 FH Errors Net %'] = (points_summary['S+1 FH Errors Net'] / points_summary['S+1 FH Errors']) * 100
points_summary['S+1 FH Errors Wide %'] = (points_summary['S+1 FH Errors Wide'] / points_summary['S+1 FH Errors']) * 100
points_summary['S+1 FH Errors Long %'] = (points_summary['S+1 FH Errors Long'] / points_summary['S+1 FH Errors']) * 100

# Display the updated summary with FH Errors categorization
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'S+1 FH Errors', 'S+1 FH Errors Net', 'S+1 FH Errors Net %', 
                 'S+1 FH Errors Wide', 'S+1 FH Errors Wide %', 'S+1 FH Errors Long', 'S+1 FH Errors Long %']]


,Player Name,S+1 FH Errors,S+1 FH Errors Net,S+1 FH Errors Net %,S+1 FH Errors Wide,S+1 FH Errors Wide %,S+1 FH Errors Long,S+1 FH Errors Long %
0,Eudice Chong,39,11,28.205128,10,25.641026,18,46.153846
1,Leila Epstein,37,8,21.621622,8,21.621622,21,56.756757
2,Ben Mitchell,11,2,18.181818,5,45.454545,4,36.363636
3,JT Bilski,13,5,38.461538,3,23.076923,5,38.461538
4,Brett Keeling,9,1,11.111111,1,11.111111,7,77.777778
5,Harrison Collomb,25,11,44.000000,7,28.000000,7,28.000000
6,Lane Durkin,5,4,80.000000,0,0.000000,1,20.000000
7,Sarah Youngberg,18,4,22.222222,6,33.333333,8,44.444444
8,Alexander Bernard,5,1,20.000000,1,20.000000,3,60.000000
9,Michael Zheng,8,2,25.000000,3,37.500000,3,37.500000


In [91]:
# Adjust the calculation to use total Serve +1 Actions as the denominator for error percentages
points_summary['S+1 FH Errors Net %'] = (points_summary['S+1 FH Errors Net'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors Wide %'] = (points_summary['S+1 FH Errors Wide'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 FH Errors Long %'] = (points_summary['S+1 FH Errors Long'] / points_summary['Total Serve +1 Actions']) * 100

# Display the corrected summary with FH Errors categorization and percentages relative to total Serve +1 Actions
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total Serve +1 Actions', 'S+1 FH Errors Net', 'S+1 FH Errors Net %', 
                 'S+1 FH Errors Wide', 'S+1 FH Errors Wide %', 'S+1 FH Errors Long', 'S+1 FH Errors Long %']]


,Player Name,Total Serve +1 Actions,S+1 FH Errors Net,S+1 FH Errors Net %,S+1 FH Errors Wide,S+1 FH Errors Wide %,S+1 FH Errors Long,S+1 FH Errors Long %
0,Eudice Chong,424,11,2.594340,10,2.358491,18,4.245283
1,Leila Epstein,278,8,2.877698,8,2.877698,21,7.553957
2,Ben Mitchell,188,2,1.063830,5,2.659574,4,2.127660
3,JT Bilski,214,5,2.336449,3,1.401869,5,2.336449
4,Brett Keeling,157,1,0.636943,1,0.636943,7,4.458599
5,Harrison Collomb,161,11,6.832298,7,4.347826,7,4.347826
6,Lane Durkin,113,4,3.539823,0,0.000000,1,0.884956
7,Sarah Youngberg,123,4,3.252033,6,4.878049,8,6.504065
8,Alexander Bernard,125,1,0.800000,1,0.800000,3,2.400000
9,Michael Zheng,118,2,1.694915,3,2.542373,3,2.542373


In [92]:
updated_summary

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,S+1 FH Winners,S+1 FH Errors,S+1 FH Winners %,S+1 FH Errors %,S+1 FH Errors Net,S+1 FH Errors Wide,S+1 FH Errors Long,S+1 FH Errors Net %,S+1 FH Errors Wide %,S+1 FH Errors Long %
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,11,39,2,9,11,10,18,2.594340,2.358491,4.245283
1,Leila Epstein,749,283,210,15,53,278,5,19,31,6,37,2,13,8,8,21,2.877698,2.877698,7.553957
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,0,11,0,5,2,5,4,1.063830,2.659574,2.127660
3,JT Bilski,572,216,175,13,26,214,6,12,26,7,13,3,6,5,3,5,2.336449,1.401869,2.336449
4,Brett Keeling,489,162,126,12,19,157,7,12,19,5,9,3,5,1,1,7,0.636943,0.636943,4.458599
5,Harrison Collomb,435,162,114,4,43,161,2,26,26,2,25,1,15,11,7,7,6.832298,4.347826,4.347826
6,Lane Durkin,363,116,95,6,12,113,5,10,26,4,5,3,4,4,0,1,3.539823,0.000000,0.884956
7,Sarah Youngberg,359,123,98,4,21,123,3,17,24,2,18,1,14,4,6,8,3.252033,4.878049,6.504065
8,Alexander Bernard,326,128,99,16,10,125,12,8,26,5,5,4,4,1,1,3,0.800000,0.800000,2.400000
9,Michael Zheng,322,121,96,7,15,118,5,12,15,0,8,0,6,2,3,3,1.694915,2.542373,2.542373


In [93]:
# Filter data for S+1 BH Winners and total BH Errors
bh_winners = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Backhand') &
    (filtered_data['C2: Serve +1 Data'] == 'Winner')
]['A1: Server'].value_counts().rename('S+1 BH Winners')

bh_errors = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Backhand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error')
]['A1: Server'].value_counts().rename('S+1 BH Errors')

# Filter data for S+1 BH Errors into the net, wide, and long based on Serve +1 columns
bh_errors_net = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Backhand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Net', na=False))
]['A1: Server'].value_counts().rename('S+1 BH Errors Net')

bh_errors_wide = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Backhand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Wide', na=False))
]['A1: Server'].value_counts().rename('S+1 BH Errors Wide')

bh_errors_long = filtered_data[
    (filtered_data['A1: Server'].isin(players)) &
    (filtered_data['C1: Serve +1 Stroke'] == 'Backhand') &
    (filtered_data['C2: Serve +1 Data'] == 'Error') &
    (filtered_data['C3: Serve +1 Placement'].str.contains('Long', na=False))
]['A1: Server'].value_counts().rename('S+1 BH Errors Long')

# Add these counts to the summary table and calculate percentages
points_summary['S+1 BH Winners'] = bh_winners
points_summary['S+1 BH Errors'] = bh_errors
points_summary['S+1 BH Errors Net'] = bh_errors_net
points_summary['S+1 BH Errors Wide'] = bh_errors_wide
points_summary['S+1 BH Errors Long'] = bh_errors_long

points_summary.fillna(0, inplace=True)
points_summary = points_summary.astype(int)

# Calculate percentages relative to total serve +1 actions
points_summary['S+1 BH Winners %'] = (points_summary['S+1 BH Winners'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 BH Errors %'] = (points_summary['S+1 BH Errors'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 BH Errors Net %'] = (points_summary['S+1 BH Errors Net'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 BH Errors Wide %'] = (points_summary['S+1 BH Errors Wide'] / points_summary['Total Serve +1 Actions']) * 100
points_summary['S+1 BH Errors Long %'] = (points_summary['S+1 BH Errors Long'] / points_summary['Total Serve +1 Actions']) * 100

# Display the updated summary with BH Winners and Errors
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total Serve +1 Actions', 'S+1 BH Winners', 'S+1 BH Winners %', 
                 'S+1 BH Errors', 'S+1 BH Errors %', 'S+1 BH Errors Net', 'S+1 BH Errors Net %', 
                 'S+1 BH Errors Wide', 'S+1 BH Errors Wide %', 'S+1 BH Errors Long', 'S+1 BH Errors Long %']]


,Player Name,Total Serve +1 Actions,S+1 BH Winners,S+1 BH Winners %,S+1 BH Errors,S+1 BH Errors %,S+1 BH Errors Net,S+1 BH Errors Net %,S+1 BH Errors Wide,S+1 BH Errors Wide %,S+1 BH Errors Long,S+1 BH Errors Long %
0,Eudice Chong,424,6,1.415094,16,3.773585,8,1.886792,3,0.707547,5,1.179245
1,Leila Epstein,278,0,0.000000,11,3.956835,9,3.237410,0,0.000000,2,0.719424
2,Ben Mitchell,188,0,0.000000,1,0.531915,0,0.000000,1,0.531915,0,0.000000
3,JT Bilski,214,0,0.000000,10,4.672897,3,1.401869,4,1.869159,3,1.401869
4,Brett Keeling,157,0,0.000000,7,4.458599,4,2.547771,2,1.273885,1,0.636943
5,Harrison Collomb,161,0,0.000000,6,3.726708,1,0.621118,1,0.621118,4,2.484472
6,Lane Durkin,113,2,1.769912,6,5.309735,3,2.654867,1,0.884956,2,1.769912
7,Sarah Youngberg,123,0,0.000000,3,2.439024,1,0.813008,2,1.626016,0,0.000000
8,Alexander Bernard,125,1,0.800000,3,2.400000,1,0.800000,1,0.800000,1,0.800000
9,Michael Zheng,118,0,0.000000,4,3.389831,1,0.847458,2,1.694915,1,0.847458


In [94]:
updated_summary

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,S+1 BH Winners,S+1 BH Errors,S+1 BH Errors Net,S+1 BH Errors Wide,S+1 BH Errors Long,S+1 BH Winners %,S+1 BH Errors %,S+1 BH Errors Net %,S+1 BH Errors Wide %,S+1 BH Errors Long %
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,6,16,8,3,5,1.415094,3.773585,1.886792,0.707547,1.179245
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,0,11,9,0,2,0.000000,3.956835,3.237410,0.000000,0.719424
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,0,1,0,1,0,0.000000,0.531915,0.000000,0.531915,0.000000
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,0,10,3,4,3,0.000000,4.672897,1.401869,1.869159,1.401869
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,0,7,4,2,1,0.000000,4.458599,2.547771,1.273885,0.636943
5,Harrison Collomb,435,162,114,4,43,161,2,26,26,...,0,6,1,1,4,0.000000,3.726708,0.621118,0.621118,2.484472
6,Lane Durkin,363,116,95,6,12,113,5,10,26,...,2,6,3,1,2,1.769912,5.309735,2.654867,0.884956,1.769912
7,Sarah Youngberg,359,123,98,4,21,123,3,17,24,...,0,3,1,2,0,0.000000,2.439024,0.813008,1.626016,0.000000
8,Alexander Bernard,326,128,99,16,10,125,12,8,26,...,1,3,1,1,1,0.800000,2.400000,0.800000,0.800000,0.800000
9,Michael Zheng,322,121,96,7,15,118,5,12,15,...,0,4,1,2,1,0.000000,3.389831,0.847458,1.694915,0.847458


In [95]:
# Ensuring we handle NaN values correctly in the 'K1: Final Shot Player' column
final_shot_data = filtered_data.dropna(subset=['K1: Final Shot Player'])

# Filter and count Final Shot Winners and Errors with corrected checks
final_shot_winners = final_shot_data[
    final_shot_data['K1: Final Shot Player'].str.contains('Winner') & 
    final_shot_data.apply(lambda x: x['A1: Server'] in x['K1: Final Shot Player'], axis=1)
]['A1: Server'].value_counts().rename('Final Shot Winners')

final_shot_errors = final_shot_data[
    final_shot_data['K1: Final Shot Player'].str.contains('Error') & 
    final_shot_data.apply(lambda x: x['A1: Server'] in x['K1: Final Shot Player'], axis=1)
]['A1: Server'].value_counts().rename('Final Shot Errors')

# Update the summary table
points_summary['Final Shot Winners'] = final_shot_winners.reindex(points_summary.index).fillna(0).astype(int)
points_summary['Final Shot Errors'] = final_shot_errors.reindex(points_summary.index).fillna(0).astype(int)

# Display the updated summary
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Final Shot Winners', 'Final Shot Errors']]


,Player Name,Final Shot Winners,Final Shot Errors
0,Eudice Chong,81,153
1,Leila Epstein,43,119
2,Ben Mitchell,33,58
3,JT Bilski,48,75
4,Brett Keeling,28,62
5,Harrison Collomb,26,77
6,Lane Durkin,23,34
7,Sarah Youngberg,17,41
8,Alexander Bernard,35,34
9,Michael Zheng,23,44


In [96]:
# Calculate FS Winner % as (Final Shot Winners / (Final Shot Winners + Final Shot Errors)) * 100
total_final_shots = points_summary['Final Shot Winners'] + points_summary['Final Shot Errors']
points_summary['FS Winner %'] = (points_summary['Final Shot Winners'] / total_final_shots) * 100

# Display the updated summary with FS Winner %
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Final Shot Winners', 'Final Shot Errors', 'FS Winner %']]


,Player Name,Final Shot Winners,Final Shot Errors,FS Winner %
0,Eudice Chong,81,153,34.615385
1,Leila Epstein,43,119,26.543210
2,Ben Mitchell,33,58,36.263736
3,JT Bilski,48,75,39.024390
4,Brett Keeling,28,62,31.111111
5,Harrison Collomb,26,77,25.242718
6,Lane Durkin,23,34,40.350877
7,Sarah Youngberg,17,41,29.310345
8,Alexander Bernard,35,34,50.724638
9,Michael Zheng,23,44,34.328358


In [97]:
# Handle NaN values before applying operations and extract player names
filtered_data['Final Shot Player Cleaned'] = filtered_data['K1: Final Shot Player'].fillna('').apply(lambda x: ' '.join(x.split(' ')[1:]))

# Filter data to find rows where the final shot was by the server and was a forehand
final_shots_forehand_data = filtered_data[
    (filtered_data['K2: Final Shot Stroke'].str.contains('Forehand', na=False)) &
    (filtered_data.apply(lambda x: x['A1: Server'] in x['Final Shot Player Cleaned'], axis=1))
]

# Count all final shots that were forehands by each server
final_shot_forehands = final_shots_forehand_data['A1: Server'].value_counts().rename('Total FS FH')

# Reindex to include all players and fill missing values with 0
final_shot_forehands = final_shot_forehands.reindex(players, fill_value=0)

# Add this new count to the summary table
points_summary['Total FS FH'] = final_shot_forehands

# Display the updated summary with Total FS FH counts
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total FS FH']]


/var/folders/hz/jkmqh6017hb_3wwh71glzdrc0000gn/T/ipykernel_7890/3573496234.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['Final Shot Player Cleaned'] = filtered_data['K1: Final Shot Player'].fillna('').apply(lambda x: ' '.join(x.split(' ')[1:]))


,Player Name,Total FS FH
0,Eudice Chong,125
1,Leila Epstein,81
2,Ben Mitchell,42
3,JT Bilski,61
4,Brett Keeling,44
5,Harrison Collomb,56
6,Lane Durkin,31
7,Sarah Youngberg,33
8,Alexander Bernard,37
9,Michael Zheng,30


In [98]:
# Filter data to count all final shots by each player
total_final_shots = filtered_data['Final Shot Player Cleaned'].value_counts().rename('Total FS')

# Filter data to count all final shots that were forehands by each player
total_final_shots_forehand = filtered_data[
    filtered_data['K2: Final Shot Stroke'].str.contains('Forehand', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('Total FS FH')

# Reindex to include all players and fill missing values with 0
total_final_shots = total_final_shots.reindex(players, fill_value=0)
total_final_shots_forehand = total_final_shots_forehand.reindex(players, fill_value=0)

# Update the summary table
points_summary['Total FS'] = total_final_shots
points_summary['Total FS FH'] = total_final_shots_forehand

# Display the updated summary with Total FS and Total FS FH counts
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total FS', 'Total FS FH']]


,Player Name,Total FS,Total FS FH
0,Eudice Chong,379,195
1,Leila Epstein,278,133
2,Ben Mitchell,179,76
3,JT Bilski,228,123
4,Brett Keeling,151,71
5,Harrison Collomb,173,90
6,Lane Durkin,106,52
7,Sarah Youngberg,116,62
8,Alexander Bernard,134,77
9,Michael Zheng,112,56


In [99]:
# Count Final Shot Winners for each player across all their shots
final_shot_winners_all = filtered_data[
    filtered_data['K1: Final Shot Player'].str.contains('Winner', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('Final Shot Winners')

# Count Final Shot Errors for each player across all their shots
final_shot_errors_all = filtered_data[
    filtered_data['K1: Final Shot Player'].str.contains('Error', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('Final Shot Errors')

# Reindex to include all players and fill missing values with 0
final_shot_winners_all = final_shot_winners_all.reindex(players, fill_value=0)
final_shot_errors_all = final_shot_errors_all.reindex(players, fill_value=0)

# Update the summary table
points_summary['Final Shot Winners'] = final_shot_winners_all
points_summary['Final Shot Errors'] = final_shot_errors_all

# Calculate FS Winner % as (Final Shot Winners / (Final Shot Winners + Final Shot Errors)) * 100
total_final_shots_all = points_summary['Final Shot Winners'] + points_summary['Final Shot Errors']
points_summary['FS Winner %'] = (points_summary['Final Shot Winners'] / total_final_shots_all) * 100

# Display the updated summary with Final Shot Winners, Errors, and FS Winner %
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Final Shot Winners', 'Final Shot Errors', 'FS Winner %']]


,Player Name,Final Shot Winners,Final Shot Errors,FS Winner %
0,Eudice Chong,114,265,30.079156
1,Leila Epstein,78,200,28.057554
2,Ben Mitchell,56,123,31.284916
3,JT Bilski,78,150,34.210526
4,Brett Keeling,42,109,27.814570
5,Harrison Collomb,36,137,20.809249
6,Lane Durkin,34,72,32.075472
7,Sarah Youngberg,39,77,33.620690
8,Alexander Bernard,61,73,45.522388
9,Michael Zheng,40,72,35.714286


In [100]:
updated_summary

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,S+1 BH Winners %,S+1 BH Errors %,S+1 BH Errors Net %,S+1 BH Errors Wide %,S+1 BH Errors Long %,Final Shot Winners,Final Shot Errors,FS Winner %,Total FS FH,Total FS
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,1.415094,3.773585,1.886792,0.707547,1.179245,114,265,30.079156,195,379
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,0.000000,3.956835,3.237410,0.000000,0.719424,78,200,28.057554,133,278
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,0.000000,0.531915,0.000000,0.531915,0.000000,56,123,31.284916,76,179
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,0.000000,4.672897,1.401869,1.869159,1.401869,78,150,34.210526,123,228
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,0.000000,4.458599,2.547771,1.273885,0.636943,42,109,27.814570,71,151
5,Harrison Collomb,435,162,114,4,43,161,2,26,26,...,0.000000,3.726708,0.621118,0.621118,2.484472,36,137,20.809249,90,173
6,Lane Durkin,363,116,95,6,12,113,5,10,26,...,1.769912,5.309735,2.654867,0.884956,1.769912,34,72,32.075472,52,106
7,Sarah Youngberg,359,123,98,4,21,123,3,17,24,...,0.000000,2.439024,0.813008,1.626016,0.000000,39,77,33.620690,62,116
8,Alexander Bernard,326,128,99,16,10,125,12,8,26,...,0.800000,2.400000,0.800000,0.800000,0.800000,61,73,45.522388,77,134
9,Michael Zheng,322,121,96,7,15,118,5,12,15,...,0.000000,3.389831,0.847458,1.694915,0.847458,40,72,35.714286,56,112


In [101]:
# Filter data to count all final shots that were forehand winners
final_shot_forehand_winners = filtered_data[
    filtered_data['K2: Final Shot Stroke'].str.contains('Forehand', na=False) &
    filtered_data['K1: Final Shot Player'].str.contains('Winner', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS FH Winners')

# Reindex to include all players and fill missing values with 0
final_shot_forehand_winners = final_shot_forehand_winners.reindex(players, fill_value=0)

# Add this new count to the summary table
points_summary['FS FH Winners'] = final_shot_forehand_winners

# Calculate FS FH Winner % as (FS FH Winners / Total FS FH) * 100
points_summary['FS FH Winner %'] = (points_summary['FS FH Winners'] / points_summary['Total FS FH']) * 100

# Display the updated summary with FS FH Winners and FS FH Winner %
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total FS FH', 'FS FH Winners', 'FS FH Winner %']]


,Player Name,Total FS FH,FS FH Winners,FS FH Winner %
0,Eudice Chong,195,65,33.333333
1,Leila Epstein,133,36,27.067669
2,Ben Mitchell,76,23,30.263158
3,JT Bilski,123,39,31.707317
4,Brett Keeling,71,18,25.352113
5,Harrison Collomb,90,15,16.666667
6,Lane Durkin,52,22,42.307692
7,Sarah Youngberg,62,17,27.419355
8,Alexander Bernard,77,33,42.857143
9,Michael Zheng,56,15,26.785714


In [102]:
# Filter data for final shots that were forehands
final_shots_forehand = filtered_data[
    filtered_data['K2: Final Shot Stroke'].str.contains('Forehand', na=False)
]

# Count FS FH Net, Long, and Wide
fs_fh_net = final_shots_forehand[
    final_shots_forehand['E4: Last Shot Placement'].str.contains('Net', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS FH Net')

fs_fh_long = final_shots_forehand[
    final_shots_forehand['E4: Last Shot Placement'].str.contains('Long', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS FH Long')

fs_fh_wide = final_shots_forehand[
    final_shots_forehand['E4: Last Shot Placement'].str.contains('Wide', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS FH Wide')

# Reindex to include all players and fill missing values with 0
fs_fh_net = fs_fh_net.reindex(players, fill_value=0)
fs_fh_long = fs_fh_long.reindex(players, fill_value=0)
fs_fh_wide = fs_fh_wide.reindex(players, fill_value=0)

# Add these counts to the summary table
points_summary['FS FH Net'] = fs_fh_net
points_summary['FS FH Long'] = fs_fh_long
points_summary['FS FH Wide'] = fs_fh_wide

# Calculate percentages relative to total FS FH
points_summary['FS FH Net %'] = (points_summary['FS FH Net'] / points_summary['Total FS FH']) * 100
points_summary['FS FH Long %'] = (points_summary['FS FH Long'] / points_summary['Total FS FH']) * 100
points_summary['FS FH Wide %'] = (points_summary['FS FH Wide'] / points_summary['Total FS FH']) * 100

# Display the updated summary with FS FH placement counts and percentages
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'Total FS FH', 'FS FH Net', 'FS FH Net %', 'FS FH Long', 'FS FH Long %', 'FS FH Wide', 'FS FH Wide %']]


,Player Name,Total FS FH,FS FH Net,FS FH Net %,FS FH Long,FS FH Long %,FS FH Wide,FS FH Wide %
0,Eudice Chong,195,28,14.358974,32,16.410256,12,6.153846
1,Leila Epstein,133,19,14.285714,17,12.781955,11,8.270677
2,Ben Mitchell,76,11,14.473684,6,7.894737,10,13.157895
3,JT Bilski,123,23,18.699187,20,16.260163,9,7.317073
4,Brett Keeling,71,11,15.492958,16,22.535211,10,14.084507
5,Harrison Collomb,90,22,24.444444,10,11.111111,3,3.333333
6,Lane Durkin,52,9,17.307692,6,11.538462,5,9.615385
7,Sarah Youngberg,62,2,3.225806,11,17.741935,2,3.225806
8,Alexander Bernard,77,14,18.181818,9,11.688312,9,11.688312
9,Michael Zheng,56,13,23.214286,8,14.285714,8,14.285714


In [103]:
updated_summary

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,Total FS FH,Total FS,FS FH Winners,FS FH Winner %,FS FH Net,FS FH Long,FS FH Wide,FS FH Net %,FS FH Long %,FS FH Wide %
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,195,379,65,33.333333,28,32,12,14.358974,16.410256,6.153846
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,133,278,36,27.067669,19,17,11,14.285714,12.781955,8.270677
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,76,179,23,30.263158,11,6,10,14.473684,7.894737,13.157895
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,123,228,39,31.707317,23,20,9,18.699187,16.260163,7.317073
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,71,151,18,25.352113,11,16,10,15.492958,22.535211,14.084507
5,Harrison Collomb,435,162,114,4,43,161,2,26,26,...,90,173,15,16.666667,22,10,3,24.444444,11.111111,3.333333
6,Lane Durkin,363,116,95,6,12,113,5,10,26,...,52,106,22,42.307692,9,6,5,17.307692,11.538462,9.615385
7,Sarah Youngberg,359,123,98,4,21,123,3,17,24,...,62,116,17,27.419355,2,11,2,3.225806,17.741935,3.225806
8,Alexander Bernard,326,128,99,16,10,125,12,8,26,...,77,134,33,42.857143,14,9,9,18.181818,11.688312,11.688312
9,Michael Zheng,322,121,96,7,15,118,5,12,15,...,56,112,15,26.785714,13,8,8,23.214286,14.285714,14.285714


In [104]:
# Filter data for final shots that were backhands
final_shots_backhand = filtered_data[
    filtered_data['K2: Final Shot Stroke'].str.contains('Backhand', na=False)
]

# Count FS BH Winners and Errors
fs_bh_winners = final_shots_backhand[
    filtered_data['K1: Final Shot Player'].str.contains('Winner', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS BH Winners')

fs_bh_errors = final_shots_backhand[
    filtered_data['K1: Final Shot Player'].str.contains('Error', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS BH Errors')

# Count FS BH Net, Long, and Wide
fs_bh_net = final_shots_backhand[
    final_shots_backhand['E4: Last Shot Placement'].str.contains('Net', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS BH Net')

fs_bh_long = final_shots_backhand[
    final_shots_backhand['E4: Last Shot Placement'].str.contains('Long', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS BH Long')

fs_bh_wide = final_shots_backhand[
    final_shots_backhand['E4: Last Shot Placement'].str.contains('Wide', na=False)
]['Final Shot Player Cleaned'].value_counts().rename('FS BH Wide')

# Reindex to include all players and fill missing values with 0
indices = players
fs_bh_winners = fs_bh_winners.reindex(indices, fill_value=0)
fs_bh_errors = fs_bh_errors.reindex(indices, fill_value=0)
fs_bh_net = fs_bh_net.reindex(indices, fill_value=0)
fs_bh_long = fs_bh_long.reindex(indices, fill_value=0)
fs_bh_wide = fs_bh_wide.reindex(indices, fill_value=0)

# Total FS BH for calculating percentages
total_fs_bh = fs_bh_winners + fs_bh_errors

# Calculate FS BH Winner %
fs_bh_winner_percent = (fs_bh_winners / total_fs_bh) * 100

# Add data to summary table
points_summary['FS BH Winners'] = fs_bh_winners
points_summary['FS BH Errors'] = fs_bh_errors
points_summary['FS BH Net'] = fs_bh_net
points_summary['FS BH Long'] = fs_bh_long
points_summary['FS BH Wide'] = fs_bh_wide
points_summary['FS BH Winner %'] = fs_bh_winner_percent.fillna(0)  # Fill NaN percentages with 0

# Display the updated summary with FS BH statistics
updated_summary = points_summary.reset_index()
updated_summary[['Player Name', 'FS BH Winners', 'FS BH Errors', 'FS BH Winner %', 'FS BH Net', 'FS BH Long', 'FS BH Wide']]


/var/folders/hz/jkmqh6017hb_3wwh71glzdrc0000gn/T/ipykernel_7890/3909758334.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  fs_bh_winners = final_shots_backhand[
/var/folders/hz/jkmqh6017hb_3wwh71glzdrc0000gn/T/ipykernel_7890/3909758334.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  fs_bh_errors = final_shots_backhand[


,Player Name,FS BH Winners,FS BH Errors,FS BH Winner %,FS BH Net,FS BH Long,FS BH Wide
0,Eudice Chong,18,109,14.173228,29,25,17
1,Leila Epstein,9,78,10.344828,16,15,17
2,Ben Mitchell,12,40,23.076923,12,3,14
3,JT Bilski,18,48,27.272727,15,7,6
4,Brett Keeling,4,28,12.500000,10,4,3
5,Harrison Collomb,6,45,11.764706,15,6,3
6,Lane Durkin,6,36,14.285714,10,7,2
7,Sarah Youngberg,4,21,16.000000,6,2,5
8,Alexander Bernard,7,23,23.333333,12,3,3
9,Michael Zheng,8,24,25.000000,7,9,1


In [105]:
updated_summary

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,FS FH Wide,FS FH Net %,FS FH Long %,FS FH Wide %,FS BH Winners,FS BH Errors,FS BH Net,FS BH Long,FS BH Wide,FS BH Winner %
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,12,14.358974,16.410256,6.153846,18,109,29,25,17,14.173228
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,11,14.285714,12.781955,8.270677,9,78,16,15,17,10.344828
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,10,14.473684,7.894737,13.157895,12,40,12,3,14,23.076923
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,9,18.699187,16.260163,7.317073,18,48,15,7,6,27.272727
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,10,15.492958,22.535211,14.084507,4,28,10,4,3,12.500000
5,Harrison Collomb,435,162,114,4,43,161,2,26,26,...,3,24.444444,11.111111,3.333333,6,45,15,6,3,11.764706
6,Lane Durkin,363,116,95,6,12,113,5,10,26,...,5,17.307692,11.538462,9.615385,6,36,10,7,2,14.285714
7,Sarah Youngberg,359,123,98,4,21,123,3,17,24,...,2,3.225806,17.741935,3.225806,4,21,6,2,5,16.000000
8,Alexander Bernard,326,128,99,16,10,125,12,8,26,...,9,18.181818,11.688312,11.688312,7,23,12,3,3,23.333333
9,Michael Zheng,322,121,96,7,15,118,5,12,15,...,8,23.214286,14.285714,14.285714,8,24,7,9,1,25.000000


In [106]:
updated_summary.columns

Index(['Player Name', 'Total Points', 'Opponent Returned Serve',
       'Serve +1 In Play', 'Serve +1 Winners', 'Serve +1 Errors',
       'Total Serve +1 Actions', 'Serve +1 Winner %', 'Serve +1 Error %',
       'Opponent Return +1 Errors', 'S+1 FH Winners', 'S+1 FH Errors',
       'S+1 FH Winners %', 'S+1 FH Errors %', 'S+1 FH Errors Net',
       'S+1 FH Errors Wide', 'S+1 FH Errors Long', 'S+1 FH Errors Net %',
       'S+1 FH Errors Wide %', 'S+1 FH Errors Long %', 'S+1 BH Winners',
       'S+1 BH Errors', 'S+1 BH Errors Net', 'S+1 BH Errors Wide',
       'S+1 BH Errors Long', 'S+1 BH Winners %', 'S+1 BH Errors %',
       'S+1 BH Errors Net %', 'S+1 BH Errors Wide %', 'S+1 BH Errors Long %',
       'Final Shot Winners', 'Final Shot Errors', 'FS Winner %', 'Total FS FH',
       'Total FS', 'FS FH Winners', 'FS FH Winner %', 'FS FH Net',
       'FS FH Long', 'FS FH Wide', 'FS FH Net %', 'FS FH Long %',
       'FS FH Wide %', 'FS BH Winners', 'FS BH Errors', 'FS BH Net',
       'FS BH L

In [107]:


# Calculate total backhand final shots as the sum of errors and winners
updated_summary['Total FS BH'] = updated_summary['FS BH Errors'] + updated_summary['FS BH Winners']

# Recalculate the error percentage columns using the total number of backhand final shots as the denominator
updated_summary['FS BH Net Error %'] = (updated_summary['FS BH Net'] / updated_summary['Total FS BH']) * 100
updated_summary['FS BH Long Error %'] = (updated_summary['FS BH Long'] / updated_summary['Total FS BH']) * 100
updated_summary['FS BH Wide Error %'] = (updated_summary['FS BH Wide'] / updated_summary['Total FS BH']) * 100

# Display the updated dataset with new calculations
updated_summary[['Player Name', 'FS BH Net Error %', 'FS BH Long Error %', 'FS BH Wide Error %']].head()


,Player Name,FS BH Net Error %,FS BH Long Error %,FS BH Wide Error %
0,Eudice Chong,22.834646,19.685039,13.385827
1,Leila Epstein,18.390805,17.241379,19.540230
2,Ben Mitchell,23.076923,5.769231,26.923077
3,JT Bilski,22.727273,10.606061,9.090909
4,Brett Keeling,31.250000,12.500000,9.375000


In [108]:
# Calculate the S+1 Effect column using the formula provided
updated_summary['S+1 Effect'] = (updated_summary['Serve +1 Winners'] + updated_summary['Opponent Return +1 Errors']) / updated_summary['Total Serve +1 Actions']*100

# Display the updated dataset with the new column
updated_summary[['Player Name', 'Serve +1 Winners', 'Opponent Return +1 Errors', 'Total Serve +1 Actions', 'S+1 Effect']].head()


,Player Name,Serve +1 Winners,Opponent Return +1 Errors,Total Serve +1 Actions,S+1 Effect
0,Eudice Chong,27,53,424,18.867925
1,Leila Epstein,15,31,278,16.546763
2,Ben Mitchell,8,34,188,22.340426
3,JT Bilski,13,26,214,18.224299
4,Brett Keeling,12,19,157,19.745223


In [109]:
updated_summary.head()

,Player Name,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,FS BH Errors,FS BH Net,FS BH Long,FS BH Wide,FS BH Winner %,Total FS BH,FS BH Net Error %,FS BH Long Error %,FS BH Wide Error %,S+1 Effect
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,109,29,25,17,14.173228,127,22.834646,19.685039,13.385827,18.867925
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,78,16,15,17,10.344828,87,18.390805,17.241379,19.540230,16.546763
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,40,12,3,14,23.076923,52,23.076923,5.769231,26.923077,22.340426
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,48,15,7,6,27.272727,66,22.727273,10.606061,9.090909,18.224299
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,28,10,4,3,12.500000,32,31.250000,12.500000,9.375000,19.745223


In [110]:
# Filter and count only 'Forehand' and 'Backhand' shots in the singles matches data
fh_bh_counts = data[data['C1: Serve +1 Stroke'].isin(['Forehand', 'Backhand'])]
fh_bh_totals = fh_bh_counts.groupby(['A1: Server', 'C1: Serve +1 Stroke']).size().unstack(fill_value=0)

# Accurately assign columns based on the types of shots
fh_bh_totals.rename(columns={
    'Forehand': 'Total S+1 FHs',
    'Backhand': 'Total S+1 BHs'
}, inplace=True)

# Ensure columns exist even if one type of shot might be missing
fh_bh_totals['Total S+1 FHs'] = fh_bh_totals.get('Total S+1 FHs', pd.Series(0, index=fh_bh_totals.index))
fh_bh_totals['Total S+1 BHs'] = fh_bh_totals.get('Total S+1 BHs', pd.Series(0, index=fh_bh_totals.index))

# Display the processed table to verify
print(fh_bh_totals.head())

C1: Serve +1 Stroke  Total S+1 BHs  Total S+1 FHs
A1: Server                                       
Aidan Harris                     7             25
Alexander Bernard               31             54
Alexandra Burnett               14             29
Amanda Hagino                   22             13
Amy Cui                         19             19


In [111]:
# Rename the column in updated_summary for matching with data
updated_summary = updated_summary.rename(columns={"Player Name": "A1: Server"})

# Merge the FH and BH counts into the updated_summary based on the server's name
updated_summary = updated_summary.merge(fh_bh_totals, on="A1: Server", how="left")

# Fill NaN values with 0 in the merged columns since not every player might have matches data
updated_summary[['Total S+1 FHs', 'Total S+1 BHs']] = updated_summary[['Total S+1 FHs', 'Total S+1 BHs']].fillna(0)

# Display the updated part of the dataset to verify the merge
updated_summary.head()

,A1: Server,Total Points,Opponent Returned Serve,Serve +1 In Play,Serve +1 Winners,Serve +1 Errors,Total Serve +1 Actions,Serve +1 Winner %,Serve +1 Error %,Opponent Return +1 Errors,...,FS BH Long,FS BH Wide,FS BH Winner %,Total FS BH,FS BH Net Error %,FS BH Long Error %,FS BH Wide Error %,S+1 Effect,Total S+1 BHs,Total S+1 FHs
0,Eudice Chong,1104,430,334,27,63,424,6,14,53,...,25,17,14.173228,127,22.834646,19.685039,13.385827,18.867925,131,252
1,Leila Epstein,749,283,210,15,53,278,5,19,31,...,15,17,10.344828,87,18.390805,17.241379,19.540230,16.546763,74,166
2,Ben Mitchell,601,191,162,8,18,188,4,9,34,...,3,14,23.076923,52,23.076923,5.769231,26.923077,22.340426,24,81
3,JT Bilski,572,216,175,13,26,214,6,12,26,...,7,6,27.272727,66,22.727273,10.606061,9.090909,18.224299,45,114
4,Brett Keeling,489,162,126,12,19,157,7,12,19,...,4,3,12.500000,32,31.250000,12.500000,9.375000,19.745223,44,91


In [112]:
# Assuming the column for forehand winners in updated_summary is named 'S+1 FH Winners'
# Check if this column exists and if 'Total S+1 FHs' is correctly calculated

# Calculate the FH Winners % as a total percentage of FHs
updated_summary['S+1 FH Winners %'] = (updated_summary['S+1 FH Winners'] / updated_summary['Total S+1 FHs']) * 100

# Fill NaN values with 0 for the case where division by zero might occur (i.e., no FHs recorded)
updated_summary['S+1 FH Winners %'] = updated_summary['S+1 FH Winners %'].fillna(0)

# Display the updated part of the dataset to verify the new column
print(updated_summary[['A1: Server', 'S+1 FH Winners', 'Total S+1 FHs', 'S+1 FH Winners %']].head())


      A1: Server  S+1 FH Winners  Total S+1 FHs  S+1 FH Winners %
0   Eudice Chong              11            252          4.365079
1  Leila Epstein               6            166          3.614458
2   Ben Mitchell               0             81          0.000000
3      JT Bilski               7            114          6.140351
4  Brett Keeling               5             91          5.494505


In [113]:
# Rename the 'A1: Server' column back to 'Player Name'
updated_summary.rename(columns={'A1: Server': 'Player Name'}, inplace=True)


In [114]:
# Calculate the Final Shot Error %
updated_summary['Final Shot Error %'] = (updated_summary['Final Shot Errors'] / updated_summary['Total FS']) * 100

# Display the first few rows to confirm the new column is added correctly
updated_summary[['Player Name', 'Total FS', 'Final Shot Errors', 'Final Shot Error %']].head()


,Player Name,Total FS,Final Shot Errors,Final Shot Error %
0,Eudice Chong,379,265,69.920844
1,Leila Epstein,278,200,71.942446
2,Ben Mitchell,179,123,68.715084
3,JT Bilski,228,150,65.789474
4,Brett Keeling,151,109,72.185430


In [115]:
# Calculate FS BH Net Error %
updated_summary['FS BH Net Error %'] = (updated_summary['FS BH Net'] / updated_summary['Total FS BH']) * 100

# Calculate FS FH Errors as the sum of FS FH Net, FS FH Long, and FS FH Wide
updated_summary['FS FH Errors'] = updated_summary['FS FH Net'] + updated_summary['FS FH Long'] + updated_summary['FS FH Wide']

# Calculate FS FH Error %
updated_summary['FS FH Error %'] = (updated_summary['FS FH Errors'] / updated_summary['Total FS FH']) * 100

# Display the first few rows to confirm the new columns are added correctly
updated_summary[['Player Name', 'FS BH Net Error %', 'FS FH Errors', 'FS FH Error %']].head()


,Player Name,FS BH Net Error %,FS FH Errors,FS FH Error %
0,Eudice Chong,22.834646,72,36.923077
1,Leila Epstein,18.390805,47,35.338346
2,Ben Mitchell,23.076923,27,35.526316
3,JT Bilski,22.727273,52,42.276423
4,Brett Keeling,31.250000,37,52.112676


In [116]:
updated_summary['FS BH Error %'] = (updated_summary['FS BH Errors'] / updated_summary['Total FS BH']) * 100

In [117]:
# Renaming the columns
updated_summary.rename(columns={
    'FS FH Net %': 'FS FH Net Error %',
    'FS FH Long %': 'FS FH Long Error %',
    'FS FH Wide %': 'FS FH Wide Error %'
}, inplace=True)

In [118]:
updated_summary.columns

Index(['Player Name', 'Total Points', 'Opponent Returned Serve',
       'Serve +1 In Play', 'Serve +1 Winners', 'Serve +1 Errors',
       'Total Serve +1 Actions', 'Serve +1 Winner %', 'Serve +1 Error %',
       'Opponent Return +1 Errors', 'S+1 FH Winners', 'S+1 FH Errors',
       'S+1 FH Winners %', 'S+1 FH Errors %', 'S+1 FH Errors Net',
       'S+1 FH Errors Wide', 'S+1 FH Errors Long', 'S+1 FH Errors Net %',
       'S+1 FH Errors Wide %', 'S+1 FH Errors Long %', 'S+1 BH Winners',
       'S+1 BH Errors', 'S+1 BH Errors Net', 'S+1 BH Errors Wide',
       'S+1 BH Errors Long', 'S+1 BH Winners %', 'S+1 BH Errors %',
       'S+1 BH Errors Net %', 'S+1 BH Errors Wide %', 'S+1 BH Errors Long %',
       'Final Shot Winners', 'Final Shot Errors', 'FS Winner %', 'Total FS FH',
       'Total FS', 'FS FH Winners', 'FS FH Winner %', 'FS FH Net',
       'FS FH Long', 'FS FH Wide', 'FS FH Net Error %', 'FS FH Long Error %',
       'FS FH Wide Error %', 'FS BH Winners', 'FS BH Errors', 'FS BH Net

In [119]:
# Calculate the raw_s+1 column
updated_summary['Raw S+1'] = updated_summary['Serve +1 Winners'] - updated_summary['Serve +1 Errors'] + updated_summary['Opponent Return +1 Errors']

# Calculate the raw/total column
updated_summary['Raw/Total'] = updated_summary['Raw S+1'] / updated_summary['Serve +1 In Play']

# Define the desired column order as a list
new_column_order = [
    'Player Name', 'Total Points', 'Opponent Returned Serve', 'Total Serve +1 Actions',
    'Serve +1 In Play', 'Serve +1 Winners', 'Serve +1 Winner %', 'Serve +1 Errors',  'Serve +1 Error %',
    
    'Opponent Return +1 Errors', 'S+1 Effect',
    
    'Total S+1 FHs', 'S+1 FH Winners','S+1 FH Winners %', 'S+1 FH Errors',  
    'S+1 FH Errors %', 'S+1 FH Errors Net', 'S+1 FH Errors Net %','S+1 FH Errors Wide',  'S+1 FH Errors Wide %', 
    'S+1 FH Errors Long', 'S+1 FH Errors Long %', 
    
    'Total S+1 BHs', 'S+1 BH Winners', 'S+1 BH Winners %','S+1 BH Errors',  'S+1 BH Errors %', 
    'S+1 BH Errors Net', 'S+1 BH Errors Net %','S+1 BH Errors Wide', 'S+1 BH Errors Wide %', 'S+1 BH Errors Long', 'S+1 BH Errors Long %',
    
    'Total FS','Final Shot Winners', 'FS Winner %', 'Final Shot Errors', 'Final Shot Error %',
    'Total FS FH', 'FS FH Winners', 'FS FH Winner %','FS FH Errors', 'FS FH Error %', 'FS FH Net', 'FS FH Net Error %', 'FS FH Long', 'FS FH Long Error %', 'FS FH Wide', 'FS FH Wide Error %',  
    'Total FS BH','FS BH Winners', 'FS BH Winner %', 'FS BH Errors', 'FS BH Error %', 'FS BH Net','FS BH Net Error %', 'FS BH Long', 'FS BH Long Error %', 'FS BH Wide',  'FS BH Wide Error %',
]

# Check for missing columns
missing_columns = [col for col in new_column_order if col not in updated_summary.columns]
print("Missing columns:", missing_columns)

# Remove missing columns from the desired column order
adjusted_column_order = [col for col in new_column_order if col in updated_summary.columns]

# Reorder the columns in updated_summary according to the adjusted order
updated_summary = updated_summary[adjusted_column_order]

updated_summary.head()


In [120]:
# Define the desired column order as a list
new_column_order = [
    'Player Name', 'Total Points', 'Opponent Returned Serve', 'Total Serve +1 Actions',
    'Serve +1 In Play', 'Serve +1 Winners', 'Serve +1 Winner %', 'Serve +1 Errors',  'Serve +1 Error %',
    
    'Opponent Return +1 Errors', 'S+1 Effect',  'Raw S+1', 'Raw/Total',
    
    'Total S+1 FHs', 'S+1 FH Winners','S+1 FH Winners %', 'S+1 FH Errors',  
    'S+1 FH Errors %', 'S+1 FH Errors Net', 'S+1 FH Errors Net %','S+1 FH Errors Wide',  'S+1 FH Errors Wide %', 
    'S+1 FH Errors Long', 'S+1 FH Errors Long %', 
    
    'Total S+1 BHs', 'S+1 BH Winners', 'S+1 BH Winners %','S+1 BH Errors',  'S+1 BH Errors %', 
    'S+1 BH Errors Net', 'S+1 BH Errors Net %','S+1 BH Errors Wide', 'S+1 BH Errors Wide %', 'S+1 BH Errors Long', 'S+1 BH Errors Long %',
    
    'Total FS','Final Shot Winners', 'FS Winner %', 'Final Shot Errors', 'Final Shot Error %',
    'Total FS FH', 'FS FH Winners', 'FS FH Winner %','FS FH Errors', 'FS FH Error %', 'FS FH Net', 'FS FH Net Error %', 'FS FH Long', 'FS FH Long Error %', 'FS FH Wide', 'FS FH Wide Error %',  
    'Total FS BH','FS BH Winners', 'FS BH Winner %', 'FS BH Errors', 'FS BH Error %', 'FS BH Net','FS BH Net Error %', 'FS BH Long', 'FS BH Long Error %', 'FS BH Wide',  'FS BH Wide Error %',
]

# Check for missing columns
missing_columns = [col for col in new_column_order if col not in updated_summary.columns]
print("Missing columns:", missing_columns)

# Remove missing columns from the desired column order
adjusted_column_order = [col for col in new_column_order if col in updated_summary.columns]

# Reorder the columns in updated_summary according to the adjusted order
updated_summary = updated_summary[adjusted_column_order]

updated_summary.head()


Missing columns: []


,Player Name,Total Points,Opponent Returned Serve,Total Serve +1 Actions,Serve +1 In Play,Serve +1 Winners,Serve +1 Winner %,Serve +1 Errors,Serve +1 Error %,Opponent Return +1 Errors,...,FS BH Winners,FS BH Winner %,FS BH Errors,FS BH Error %,FS BH Net,FS BH Net Error %,FS BH Long,FS BH Long Error %,FS BH Wide,FS BH Wide Error %
0,Eudice Chong,1104,430,424,334,27,6,63,14,53,...,18,14.173228,109,85.826772,29,22.834646,25,19.685039,17,13.385827
1,Leila Epstein,749,283,278,210,15,5,53,19,31,...,9,10.344828,78,89.655172,16,18.390805,15,17.241379,17,19.540230
2,Ben Mitchell,601,191,188,162,8,4,18,9,34,...,12,23.076923,40,76.923077,12,23.076923,3,5.769231,14,26.923077
3,JT Bilski,572,216,214,175,13,6,26,12,26,...,18,27.272727,48,72.727273,15,22.727273,7,10.606061,6,9.090909
4,Brett Keeling,489,162,157,126,12,7,19,12,19,...,4,12.500000,28,87.500000,10,31.250000,4,12.500000,3,9.375000


In [121]:
updated_summary = updated_summary.round(2)

In [122]:
# save the DataFrame to a CSV file (for verification or local use)
updated_summary.to_csv('/Users/lkimball/Desktop/Wesleyan_Tennis_Analytics/S+1_df.csv', index=True)